# Web Scraping in Practice

## Introduction

Scraping a real website is an iterative process: inspect the page, select a container, drill down, clean the data, repeat. This notebook walks through a complete pipeline on [books.toscrape.com](http://books.toscrape.com) — a site designed for scraping practice — covering the inspect-element workflow, container selection, regex-based class matching, pagination, and image downloading.

## Objectives

You will be able to:

- Use browser DevTools to identify the HTML structure of a target element
- Select a page container and drill down to specific fields
- Use regular expressions with `find_all` to match class name patterns
- Paginate over multiple pages using URL patterns
- Download and display images scraped from a web page

In [ ]:
from bs4 import BeautifulSoup
import requests
import re
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import shutil
import os
%matplotlib inline

---

## Step 1 — Fetch the Page

In [ ]:
html_page = requests.get('http://books.toscrape.com/')
print('Status:', html_page.status_code)

soup = BeautifulSoup(html_page.content, 'html.parser')

---

## Step 2 — Use Inspect Element to Find Your Container

Right-click any book on the page and choose **Inspect**. You'll see the book listings live inside a `<section>` tag. Just above the section is a warning `<div>` that's easy to locate programmatically:

<img src="assets/web_scraping_in_practice/inspect.png" width="900">

<img src="assets/web_scraping_in_practice/book-section.png" width="800">

In [ ]:
# Locate the warning div — it's a reliable landmark just above the books section
warning = soup.find('div', class_='alert alert-warning')
warning

In [ ]:
# Move to the next sibling element — the book container section
# (need two calls because whitespace text nodes sit between tags)
book_container = warning.nextSibling.nextSibling
print(type(book_container), book_container.name)

---

## Step 3 — Extract Titles

Each book title lives inside an `<h3>` tag, within an `<a>` tag whose `title` attribute holds the full title (the visible text is truncated).

In [ ]:
# Preview the first h3
book_container.find('h3')

In [ ]:
# Drill down: h3 -> a -> title attribute
book_container.find('h3').find('a').attrs['title']

In [ ]:
# Extract all 20 titles from this page
titles = [h3.find('a').attrs['title'] for h3 in book_container.find_all('h3')]
print(len(titles), titles[:5])

---

## Step 4 — Extract Star Ratings with Regex

Star ratings are encoded in the CSS class: `<p class="star-rating Three">`. Since `find_all` accepts a compiled regex, we can match any class that starts with `"star-rating "`.

In [ ]:
# The class name is 'star-rating One/Two/Three/Four/Five'
# Regex matches the pattern; the last element of the class list is the word rating
regex = re.compile('star-rating (.*)')

rating_tags = book_container.find_all('p', {'class': regex})
print('Found', len(rating_tags), 'ratings')
rating_tags[0]  # preview

In [ ]:
# Extract word rating, then convert to integer
word_to_int = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}

star_ratings = [word_to_int[p.attrs['class'][-1]] for p in rating_tags]
print(star_ratings)

---

## Step 5 — Extract Prices and Availability

<img src="assets/web_scraping_in_practice/book_img.png" width="800">

In [ ]:
# Prices — class 'price_color', strip the £ symbol
price_tags = book_container.find_all('p', class_='price_color')
prices = [float(p.text[1:]) for p in price_tags]  # [1:] removes £
print(prices[:5])

In [ ]:
# Availability — class 'instock availability'
avail_tags = book_container.find_all('p', class_='instock availability')
availabilities = [a.text.strip() for a in avail_tags]
print(availabilities[:5])

---

## Step 6 — Build the DataFrame

In [ ]:
df = pd.DataFrame({
    'title':        titles,
    'star_rating':  star_ratings,
    'price_gbp':    prices,
    'availability': availabilities,
})
print(df.shape)
df.head()

---

## Step 7 — Pagination

The site has 50 pages. The URL pattern is predictable:

- Page 1: `http://books.toscrape.com/`
- Page 2: `http://books.toscrape.com/catalogue/page-2.html`
- Page 3: `http://books.toscrape.com/catalogue/page-3.html`

Wrap your extraction logic in helper functions, then loop.

In [ ]:
def get_book_container(soup):
    warning = soup.find('div', class_='alert alert-warning')
    return warning.nextSibling.nextSibling

def retrieve_titles(container):
    return [h3.find('a').attrs['title'] for h3 in container.find_all('h3')]

def retrieve_ratings(container):
    word_to_int = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}
    regex = re.compile('star-rating (.*)')
    return [word_to_int[p.attrs['class'][-1]]
            for p in container.find_all('p', {'class': regex})]

def retrieve_prices(container):
    return [float(p.text[1:]) for p in container.find_all('p', class_='price_color')]

def retrieve_availabilities(container):
    return [a.text.strip() for a in container.find_all('p', class_='instock availability')]

In [ ]:
# Scrape all 50 pages — requires internet connection
import time

all_titles, all_ratings, all_prices, all_avails = [], [], [], []

# Page 1 already fetched
container = get_book_container(soup)
all_titles      += retrieve_titles(container)
all_ratings     += retrieve_ratings(container)
all_prices      += retrieve_prices(container)
all_avails      += retrieve_availabilities(container)

# Pages 2–50
for i in range(2, 51):
    url = f'http://books.toscrape.com/catalogue/page-{i}.html'
    page = requests.get(url)
    page_soup = BeautifulSoup(page.content, 'html.parser')
    container = get_book_container(page_soup)
    all_titles      += retrieve_titles(container)
    all_ratings     += retrieve_ratings(container)
    all_prices      += retrieve_prices(container)
    all_avails      += retrieve_availabilities(container)
    time.sleep(0.2)  # be polite — 5 requests/second max

full_df = pd.DataFrame({
    'title':        all_titles,
    'star_rating':  all_ratings,
    'price_gbp':    all_prices,
    'availability': all_avails,
})
print(full_df.shape)  # should be (1000, 4)
full_df.head()

---

## Step 8 — Scraping Images

Book cover images are `<img>` tags. Their `src` attribute gives a relative URL; prepend the base URL to build the full download link.

In [ ]:
# Re-fetch page 1 if needed
html_page = requests.get('http://books.toscrape.com/')
soup = BeautifulSoup(html_page.content, 'html.parser')
container = get_book_container(soup)

# Find all img tags
images = container.find_all('img')
print(images[0])  # preview

In [ ]:
# Build full URL from relative src
url_base = 'http://books.toscrape.com/'
ex_img = images[0]
full_url = url_base + ex_img.attrs['src']
print(full_url)

In [ ]:
# Download a single image
os.makedirs('assets/scraping_images', exist_ok=True)

r = requests.get(full_url, stream=True)
if r.status_code == 200:
    with open('assets/scraping_images/book1.jpg', 'wb') as f:
        r.raw.decode_content = True
        shutil.copyfileobj(r.raw, f)
    print('Saved.')

In [ ]:
# Display it with matplotlib
img = mpimg.imread('assets/scraping_images/book1.jpg')
plt.imshow(img)
plt.axis('off')
plt.title(ex_img.attrs['alt'])
plt.show()

In [ ]:
# Download all 20 covers from page 1 and display them in a DataFrame
from IPython.display import HTML

data = []
for n, img_tag in enumerate(images):
    full_url = url_base + img_tag.attrs['src']
    path = f'assets/scraping_images/book{n+1}.jpg'
    r = requests.get(full_url, stream=True)
    if r.status_code == 200:
        with open(path, 'wb') as f:
            r.raw.decode_content = True
            shutil.copyfileobj(r.raw, f)
        data.append([img_tag.attrs['alt'], f'<img src="{path}" width="60"/>'])

covers_df = pd.DataFrame(data, columns=['title', 'cover'])
HTML(covers_df.to_html(escape=False))

---

## Practice

The functions above use URL hacking to paginate. Write an alternative version that follows the **"next" button** link instead.

In [ ]:
# Write a function that takes a soup object and returns the URL for the next page.
# Hint: look for an <li class='next'> containing an <a> tag.
# Return None if there is no next page.

def next_page_url(soup, base='http://books.toscrape.com/catalogue/'):
    # Your code here
    pass

In [ ]:
# Use your function to scrape all 50 pages without hardcoding page numbers.
# Start from http://books.toscrape.com/ and keep following next_page_url() until it returns None.


---

## Summary

In this notebook you built a complete scraping pipeline:

1. **Fetch** the page with `requests.get()` → parse with `BeautifulSoup`
2. **Locate a container** using Inspect Element — find a stable landmark tag and navigate from it
3. **Drill down** to individual fields with `find`/`find_all`, `.attrs`, and `.text.strip()`
4. **Regex matching** — pass `re.compile(pattern)` to `find_all` to match class name patterns
5. **Paginate** with URL hacking or by following the next-page link
6. **Download images** with `requests.get(stream=True)` + `shutil.copyfileobj`; display with `IPython.display.HTML`